# QSAR model performance comparison

This notebook compares Random Forest and XGBoost performance on the same QSAR train/test split. It exports per-target ROC-AUC values and a summary table with mean ROC-AUC per model.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

cwd = Path.cwd().resolve()
if cwd.name == "qsar":
    base_dir = cwd
elif (cwd / "qsar").exists():
    base_dir = (cwd / "qsar").resolve()
else:
    base_dir = cwd

if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

from qsar_common import (
    add_mol_column,
    aggregate_targets_by_fingerprint,
    build_morgan_fingerprints,
    encode_targets,
    load_qsar_dataset,
    stack_features_and_targets,
    to_target_probability_matrix,
)
from qsar_config import DATA_PATH, ECFP_N_BITS, ECFP_RADIUS, RANDOM_SEED, TEST_SIZE

In [2]:
def prepare_xy():
    df = load_qsar_dataset(DATA_PATH)
    df = add_mol_column(df, smiles_column="Smiles", mol_column="mol")
    df = build_morgan_fingerprints(
        df,
        mol_column="mol",
        output_column="fp",
        radius=ECFP_RADIUS,
        n_bits=ECFP_N_BITS,
    )
    df, _, target_names = encode_targets(
        df,
        target_column="Target",
        output_column="target_encoded",
    )
    df_agg = aggregate_targets_by_fingerprint(
        df,
        fp_column="fp",
        encoded_target_column="target_encoded",
        aggregated_target_column="target",
    )
    x, y = stack_features_and_targets(df_agg, fp_column="fp", target_column="target")
    return x, y, target_names


def compute_auc_rows(model_name, y_true, y_prob, target_names):
    rows = []
    auc_values = []

    for i, target in enumerate(target_names):
        if len(np.unique(y_true[:, i])) < 2:
            auc = np.nan
        else:
            auc = float(roc_auc_score(y_true[:, i], y_prob[:, i]))
            auc_values.append(auc)

        rows.append({"model": model_name, "target": str(target), "roc_auc": auc})

    rows.append(
        {
            "model": model_name,
            "target": "__MEAN__",
            "roc_auc": float(np.mean(auc_values)) if auc_values else np.nan,
        }
    )
    return rows

In [3]:
x, y, target_names = prepare_xy()

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)

rf_model = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
)
rf_model.fit(x_train, y_train)

xgb_model = OneVsRestClassifier(
    XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1,
    )
)
xgb_model.fit(x_train, y_train)

rf_prob = to_target_probability_matrix(rf_model.predict_proba(x_test), n_targets=y.shape[1])
xgb_prob = to_target_probability_matrix(xgb_model.predict_proba(x_test), n_targets=y.shape[1])

rows = []
rows.extend(compute_auc_rows("random_forest", y_test, rf_prob, target_names))
rows.extend(compute_auc_rows("xgboost", y_test, xgb_prob, target_names))

target_df = pd.DataFrame(rows)
summary_df = (
    target_df[target_df["target"] == "__MEAN__"]
    .rename(columns={"roc_auc": "mean_roc_auc"})
    .reset_index(drop=True)
)

out_dir = base_dir / "outputs" / "model-comparison"
out_dir.mkdir(parents=True, exist_ok=True)

target_path = out_dir / "model_performance_per_target.csv"
summary_path = out_dir / "model_performance_summary.csv"
target_df.to_csv(target_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"Saved per-target comparison: {target_path}")
print(f"Saved summary comparison: {summary_path}")
summary_df

Saved per-target comparison: /mnt/ArchData/GitHub/qspr-qsar-explainability/qsar/outputs/model-comparison/model_performance_per_target.csv
Saved summary comparison: /mnt/ArchData/GitHub/qspr-qsar-explainability/qsar/outputs/model-comparison/model_performance_summary.csv


,model,target,mean_roc_auc
0,random_forest,__MEAN__,0.981150
1,xgboost,__MEAN__,0.978076
